# 07 · Demo: try your own sentences

**Input:** the saved models in `outputs/bilstm/` and `outputs/bert/` (notebooks 04, 05). This notebook only predicts; it does not train.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

## 1. Load the two models

In [ ]:
import json

import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.config import BERT_DIR, BILSTM_DIR, CFG, DEVICE, LABELS, setup_notebook
from src.data import Vocab
from src.evaluation import predict_transformer
from src.explain import lstm_predict_texts, show_attention
from src.models import BiLSTMAttention
from src.text import clean_text

setup_notebook()

vocab = Vocab.load(BILSTM_DIR / 'vocab.json')
bilstm = BiLSTMAttention(len(vocab), CFG['lstm_emb_dim'], CFG['lstm_hidden'], CFG['lstm_layers'], 3,
                         CFG['lstm_dropout']).to(DEVICE)
bilstm.load_state_dict(torch.load(BILSTM_DIR / 'model.pt', map_location=DEVICE))
bert_tok = AutoTokenizer.from_pretrained(BERT_DIR)
bert = AutoModelForSequenceClassification.from_pretrained(BERT_DIR).to(DEVICE)
BERT_MAX_LEN = json.loads((BERT_DIR / 'train_info.json').read_text()).get('max_len', 256)

## 2. Try your own sentences

In [ ]:
demo_texts = [
    "This coffee is absolutely amazing, I will definitely buy it again!",
    "Not bad at all, actually pretty tasty.",
    "It's okay. Nothing special, but it does the job.",
    "Great, another box arrived completely crushed. Just what I needed.",
    "My dog loves these treats, but they are way too expensive for such a small bag.",
    "I would not recommend this to anyone. Total waste of money.",
]
demo_clean = [clean_text(t) for t in demo_texts]
fmt = lambda p: f'{LABELS[p.argmax()]} ({p.max():.0%})'
pd.DataFrame({'Sentence': demo_texts,
              'BiLSTM + Attention': [fmt(p) for p in lstm_predict_texts(bilstm, vocab, demo_texts, CFG['lstm_max_len'])],
              'BERT fine-tune': [fmt(p) for p in predict_transformer(bert, bert_tok, demo_clean, BERT_MAX_LEN, desc='demo')]})

In [ ]:
for t in demo_texts[1:5]:
    show_attention(bilstm, vocab, t, CFG['lstm_max_len'])